In [ ]:
%%writefile submission.py

"""
Orbit Wars — 智能体 v4（目标 750+ ELO）
========================================

物理系统全面升级
  • 类型分发器：静态行星 / 轨道行星 / 彗星预测分别处理
  • 8 次迭代拦截求解器；收敛容差 0.2（原来是 4 次迭代 / 0.5）
  • 彗星近太阳安全检测：拒绝距中心 22 单位以内的拦截目标
  • 每侧 7 个候选太阳绕行路径点（原来是 3 个）

策略全面升级
  • 移除了 taken_targets 锁定 —— 通过 committed_to 实现多源协调
  • 目标优先贪心算法：全局评分所有目标；最近来源优先填充
  • 净 ROI 评分：production × productive_turns / (needed × time_discount)
  • 竞速奖励：当敌方舰队已在途中时提升优先级
  • 反击机制：from_planet_id 可检测刚被削弱的行星
  • 战略敌人奖励：优先摧毁敌方产量最高的行星
  • 4 人 vs 2 人对战适应（中立偏好、侵略性缩放）
  • 无硬性距离上限 —— 远处高产量行星会被正确估值
  • 最小舰队规模（12 艘）以保证可行的航行速度
  • 自适应驻军：后期游戏中产量缓冲缩减
  • 彗星 ROI 门槛：尝试占领前要求剩余 ≥18 个生产回合

回合结构
  阶段 0  解析 + 分类
  阶段 1  受威胁行星紧急增援
  阶段 2  目标优先多源协调攻击
  阶段 3  二次打击（使用剩余舰船进行部分覆盖）
  阶段 4  前线集结（后方 → 前线行星）
  阶段 5  终局全力一击（最后 35 回合）
"""

import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

# ─── 引擎常量 ─────────────────────────────────────────────────────────────
CENTER_X = CENTER_Y  = 50.0
SUN_R           = 10.0      # 真实太阳碰撞半径
SUN_SAFETY      = 1.5       # 路径检查时额外增加的安全余量
MAX_SPEED       = 6.0       # 最大舰队速度（单位/回合）
ROTATION_LIMIT  = 50.0      # orbital_radius + planet_radius 的轨道判定阈值
TOTAL_STEPS     = 500

# ─── 策略常量 ───────────────────────────────────────────────────────────
MIN_KEEP         = 5         # 任意行星上始终保留的最低舰船数（硬底线）
MIN_FLEET        = 12        # 每支舰队的最低舰船数（速度底线）
BUFFER           = 3         # 在预测驻军之上额外增加的舰船数
AIM_ITERS        = 8         # 拦截求解器迭代次数
ETA_CAP          = 110       # 忽略 ETA 超过此值的舰队/舰船
PROD_RESERVE     = 9         # 作为储备保留的生产回合数（常规）
PROD_RESERVE_LATE = 4        # 同上，后期游戏
SUN_EXCL_R       = 22.0      # 拒绝目标位置距太阳此距离以内的拦截
COMET_MIN_LIFE   = 18        # 不占领剩余生产回合少于此值的彗星


# ═══════════════════════════════════════════════════════════════════════════
#  物理引擎
# ═══════════════════════════════════════════════════════════════════════════

def pdist(ax, ay, bx, by):
    return math.hypot(ax - bx, ay - by)


def fleet_speed(ships):
    """对数速度曲线 —— 与 orbit_wars 引擎完全一致。"""
    if ships <= 1:
        return 1.0
    ratio = min(1.0, math.log(max(1, ships)) / math.log(1000.0))
    return 1.0 + (MAX_SPEED - 1.0) * (ratio ** 1.5)


def segment_hits_sun(x1, y1, x2, y2, safety=SUN_SAFETY):
    """当线段 (x1,y1)→(x2,y2) 穿过太阳（含安全余量）时返回 True。"""
    r  = SUN_R + safety
    dx, dy = x2 - x1, y2 - y1
    fx, fy = x1 - CENTER_X, y1 - CENTER_Y
    a = dx * dx + dy * dy
    if a < 1e-9:
        return pdist(x1, y1, CENTER_X, CENTER_Y) < r
    b    = 2 * (fx * dx + fy * dy)
    c    = fx * fx + fy * fy - r * r
    disc = b * b - 4 * a * c
    if disc < 0:
        return False
    sq = math.sqrt(disc)
    t1 = (-b - sq) / (2 * a)
    t2 = (-b + sq) / (2 * a)
    return (0 <= t1 <= 1) or (0 <= t2 <= 1)


def safe_route(sx, sy, tx, ty):
    """
    返回从 src→dst 的太阳安全路径的 (angle, effective_distance)。
    当直线路径被阻挡时，使用 7 种安全倍率在太阳两侧尝试绕行路径点；
    选择最短的可行绕道路径。
    """
    if not segment_hits_sun(sx, sy, tx, ty):
        return math.atan2(ty - sy, tx - sx), pdist(sx, sy, tx, ty)

    vx, vy = tx - sx, ty - sy
    norm   = math.hypot(vx, vy)
    if norm < 1e-9:
        return math.atan2(ty - sy, tx - sx), pdist(sx, sy, tx, ty)

    nx, ny = -vy / norm, vx / norm      # 垂直单位向量
    best   = None

    for sign in (1.0, -1.0):
        for mult in (1.2, 1.5, 1.9, 2.4, 3.0, 4.0, 5.5):
            wx = CENTER_X + sign * nx * SUN_R * mult
            wy = CENTER_Y + sign * ny * SUN_R * mult
            if segment_hits_sun(sx, sy, wx, wy) or segment_hits_sun(wx, wy, tx, ty):
                continue
            d = pdist(sx, sy, wx, wy) + pdist(wx, wy, tx, ty)
            if best is None or d < best[0]:
                best = (d, wx, wy)
            break       # 本侧已找到最紧凑的有效路径点

    if best is None:
        # 无可行绕道路径 —— 放大距离使此路径评分较低
        return math.atan2(ty - sy, tx - sx), pdist(sx, sy, tx, ty) * 2.5

    _, wx, wy = best
    return math.atan2(wy - sy, wx - sx), best[0]


def get_arrival(sx, sy, tx, ty, ships):
    """一支 `ships` 规模的舰队从 (sx,sy) 到 (tx,ty) 的 (angle, turns_int)。"""
    angle, d = safe_route(sx, sy, tx, ty)
    return angle, max(1, int(math.ceil(d / fleet_speed(ships))))


# ─── 行星类型预测器 ───────────────────────────────────────────────────────

def _orbiting_pos(planet, initial_by_id, ang_vel, turns):
    """任意行星（静态或轨道）在 `turns` 回合后的位置。"""
    init = initial_by_id.get(planet.id)
    if init is None:
        return planet.x, planet.y
    r = pdist(init.x, init.y, CENTER_X, CENTER_Y)
    if r + init.radius >= ROTATION_LIMIT:
        return planet.x, planet.y          # 静态外圈行星
    cur = math.atan2(planet.y - CENTER_Y, planet.x - CENTER_X)
    new = cur + ang_vel * turns
    return CENTER_X + r * math.cos(new), CENTER_Y + r * math.sin(new)


def _comet_pos(pid, comets, turns):
    """彗星行星 `pid` 在 `turns` 回合后的位置。如果已过期则返回 None。"""
    for g in comets:
        pids = g.get("planet_ids", [])
        if pid not in pids:
            continue
        idx  = pids.index(pid)
        paths = g.get("paths", [])
        pi   = g.get("path_index", 0)
        if idx >= len(paths):
            return None
        path = paths[idx]
        fi   = pi + int(turns)
        return (path[fi][0], path[fi][1]) if 0 <= fi < len(path) else None
    return None


def comet_life(pid, comets):
    """彗星行星 `pid` 的剩余回合数。"""
    for g in comets:
        pids = g.get("planet_ids", [])
        if pid not in pids:
            continue
        idx   = pids.index(pid)
        paths = g.get("paths", [])
        pi    = g.get("path_index", 0)
        return max(0, len(paths[idx]) - pi) if idx < len(paths) else 0
    return 0


def predict_pos(planet, initial_by_id, ang_vel, turns, comet_ids, comets):
    """通用位置预测，按行星类型分发。"""
    if planet.id in comet_ids:
        return _comet_pos(planet.id, comets, turns)    # 可能为 None
    return _orbiting_pos(planet, initial_by_id, ang_vel, turns)


def aim_at(src, tgt, ships, initial_by_id, ang_vel, comets, comet_ids):
    """
    迭代拦截求解器。
    返回 (angle, turns, px, py)，目标不可达或不安全时返回 None。

    关键安全检查：
      • 彗星在收敛过程中过期 → None
      • 预测拦截点距太阳 SUN_EXCL_R 以内 → None
        （捕捉快速摆动靠近太阳的彗星——不值得追）
    """
    is_comet = tgt.id in comet_ids
    tol      = 0.4 if is_comet else 0.2

    tx, ty = tgt.x, tgt.y
    for _ in range(AIM_ITERS):
        angle, turns = get_arrival(src.x, src.y, tx, ty, ships)
        pos = predict_pos(tgt, initial_by_id, ang_vel, turns, comet_ids, comets)
        if pos is None:
            return None        # 彗星已过期
        ntx, nty = pos

        # 拒绝太阳排除区域内的拦截点
        if pdist(ntx, nty, CENTER_X, CENTER_Y) < SUN_EXCL_R:
            return None        # 彗星近日点太接近太阳 —— 跳过

        if abs(ntx - tx) < tol and abs(nty - ty) < tol:
            tx, ty = ntx, nty
            break
        tx, ty = ntx, nty

    angle, turns = get_arrival(src.x, src.y, tx, ty, ships)
    return angle, turns, tx, ty


# ═══════════════════════════════════════════════════════════════════════════
#  舰队跟踪（几何投影 —— 非角度比较）
# ═══════════════════════════════════════════════════════════════════════════

def fleet_target(f, planets):
    """舰队 f 正朝哪个行星飞行？返回 (planet, eta) 或 (None, None)。"""
    fvx, fvy = math.cos(f.angle), math.sin(f.angle)
    sp = fleet_speed(f.ships)
    best_p, best_t = None, 1e9
    for p in planets:
        dx, dy = p.x - f.x, p.y - f.y
        proj   = dx * fvx + dy * fvy
        if proj <= 0:
            continue
        perp = abs(dx * fvy - dy * fvx)
        if perp > p.radius + 1.2:
            continue
        t = proj / sp
        if t < best_t and t <= ETA_CAP:
            best_t, best_p = t, p
    return (best_p, int(math.ceil(best_t))) if best_p else (None, None)


def fleets_to(planet, fleets):
    """所有飞向 planet 的舰队的 [(eta, owner, ships), ...] 列表。"""
    arrivals = []
    for f in fleets:
        fvx, fvy = math.cos(f.angle), math.sin(f.angle)
        dx, dy   = planet.x - f.x, planet.y - f.y
        proj     = dx * fvx + dy * fvy
        if proj <= 0:
            continue
        perp = abs(dx * fvy - dy * fvx)
        if perp > planet.radius + 1.5:
            continue
        t = proj / fleet_speed(f.ships)
        if t > ETA_CAP:
            continue
        arrivals.append((int(math.ceil(t)), f.owner, int(f.ships)))
    return arrivals


# ═══════════════════════════════════════════════════════════════════════════
#  战斗模拟
# ═══════════════════════════════════════════════════════════════════════════

def defense_shortfall(planet, arrivals, player):
    """
    事件驱动的驻军模拟。
    返回为使此行星在所有攻击波中存活当前所需的额外舰船数。
    0 = 安全。
    """
    if not arrivals:
        return 0
    events   = sorted(arrivals)
    garrison = planet.ships
    prod     = planet.production if planet.owner == player else 0
    last_t   = 0
    deficit  = 0
    i = 0
    while i < len(events):
        t = events[i][0]
        garrison += (t - last_t) * prod
        group = []
        while i < len(events) and events[i][0] == t:
            group.append(events[i])
            i += 1
        friendly = sum(s for _, o, s in group if o == player)
        enemy    = sum(s for _, o, s in group if o != player)
        garrison += friendly - enemy
        if garrison < 0:
            deficit = max(deficit, -garrison)
        last_t = t
    return deficit


def garrison_at_arrival(tgt, eta_turns, early_enemy):
    """
    当我方舰队在 eta_turns 回合到达时目标行星的驻军状态，
    已处理在我们之前到达的敌方舰队。
    返回 (estimated_garrison, owner_then)。
    """
    if not early_enemy:
        if tgt.owner == -1:
            return tgt.ships, -1
        return tgt.ships + tgt.production * eta_turns, tgt.owner

    events   = sorted(early_enemy)          # [(eta, ships), ...]
    garrison = tgt.ships
    owner    = tgt.owner
    prod     = tgt.production
    last_t   = 0

    for t, o_ships in events:
        if owner != -1:
            garrison += (t - last_t) * prod
        if o_ships > garrison:
            owner    = -2   # 被未知敌人占领
            garrison = o_ships - garrison
        else:
            garrison -= o_ships
        last_t = t

    if owner != -1:
        garrison += (eta_turns - last_t) * prod
    return max(0, int(garrison)), owner


# ═══════════════════════════════════════════════════════════════════════════
#  主智能体
# ═══════════════════════════════════════════════════════════════════════════

def agent(obs):

    # ── 阶段 0：解析与分类 ────────────────────────────────────────────────
    get = obs.get if isinstance(obs, dict) else lambda k, d=None: getattr(obs, k, d)

    player    = get("player", 0)
    step      = get("step", 0) or 0
    ang_vel   = get("angular_velocity", 0.0) or 0.0
    raw_pl    = get("planets", []) or []
    raw_fl    = get("fleets",  []) or []
    raw_init  = get("initial_planets", []) or []
    comets    = get("comets", []) or []
    comet_ids = set(get("comet_planet_ids", []) or [])

    planets       = [Planet(*p) for p in raw_pl]
    fleets        = [Fleet(*f)  for f in raw_fl]
    initial_by_id = {Planet(*p).id: Planet(*p) for p in raw_init}

    my_planets    = [p for p in planets if p.owner == player]
    enemy_planets = [p for p in planets if p.owner not in (player, -1)]
    other_planets = [p for p in planets if p.owner != player]

    if not my_planets:
        return []

    # ── 游戏状态标志 ─────────────────────────────────────────────────────
    remaining    = max(1, TOTAL_STEPS - step)
    is_early     = step < 70
    is_late      = remaining < 80
    is_very_late = remaining < 35

    # 计算活跃玩家数（用于策略调整）
    active_owners  = {p.owner for p in planets if p.owner >= 0}
    active_owners |= {f.owner for f in fleets  if f.owner >= 0}
    multiplayer    = len(active_owners) >= 3

    # 每位玩家的舰船总数
    ships_by = {}
    for p in planets:
        if p.owner >= 0:
            ships_by[p.owner] = ships_by.get(p.owner, 0) + p.ships
    for f in fleets:
        if f.owner >= 0:
            ships_by[f.owner] = ships_by.get(f.owner, 0) + f.ships

    my_ships  = ships_by.get(player, 0)
    enemy_max = max((v for k, v in ships_by.items() if k != player), default=1)
    trailing  = my_ships < enemy_max * 0.80
    winning   = my_ships > enemy_max * 1.35

    # ── 行星类型辅助函数 ────────────────────────────────────────────────
    def is_rotating(p):
        init = initial_by_id.get(p.id)
        if init is None:
            return False
        return pdist(init.x, init.y, CENTER_X, CENTER_Y) + init.radius < ROTATION_LIMIT

    # ── 舰队跟踪 ─────────────────────────────────────────────────────────
    # committed_to[tid] = 当前飞行中飞向行星 tid 的我方舰船总数
    committed_to  = {}
    inbound_enemy = {}   # tid → [(eta, ships), ...]

    for f in fleets:
        tgt, eta = fleet_target(f, planets)
        if tgt is None:
            continue
        if f.owner == player:
            committed_to[tgt.id] = committed_to.get(tgt.id, 0) + f.ships
        else:
            inbound_enemy.setdefault(tgt.id, []).append((eta, f.ships))

    # ── 反击：刚被削弱的敌方行星 ────────────────────────────────────────
    # from_planet_id 揭示了敌方每支舰队来自哪个行星
    enemy_shipped_from = {}
    for f in fleets:
        if f.owner != player:
            fpid = getattr(f, "from_planet_id", -1)
            if fpid >= 0:
                enemy_shipped_from[fpid] = (enemy_shipped_from.get(fpid, 0)
                                            + f.ships)

    # ── 战略敌人分析 ──────────────────────────────────────────────────
    # 识别每个敌方拥有者的最高产量行星
    enemy_max_prod = {}
    for p in enemy_planets:
        enemy_max_prod[p.owner] = max(enemy_max_prod.get(p.owner, 0),
                                      p.production)

    def is_strategic(tgt):
        return (tgt.owner not in (player, -1)
                and tgt.production >= enemy_max_prod.get(tgt.owner, 0))

    # ── 竞速条件分析 ───────────────────────────────────────────────────
    def enemy_min_eta(tgt):
        """任意敌人到达 tgt 的粗略最小 ETA。"""
        m = float("inf")
        for eta_t, _ in inbound_enemy.get(tgt.id, []):
            m = min(m, eta_t)
        for ep in enemy_planets:
            d = pdist(ep.x, ep.y, tgt.x, tgt.y)
            m = min(m, d / fleet_speed(30))   # 30 艘 ≈ 合理的速度代理
        return m

    # ── 防御储备 ────────────────────────────────────────────────────────
    prod_res = PROD_RESERVE_LATE if is_late else PROD_RESERVE

    reserve = {}
    for p in my_planets:
        arrivals = fleets_to(p, fleets)
        need     = defense_shortfall(p, arrivals, player)
        prod_buf = min(p.ships // 3, p.production * prod_res)
        reserve[p.id] = min(p.ships - MIN_KEEP, max(0, need) + prod_buf)

    available = {p.id: max(0, p.ships - max(MIN_KEEP, reserve[p.id]))
                 for p in my_planets}

    moves = []

    # ── 局部辅助函数 ────────────────────────────────────────────────────
    def depart_ok(src, angle):
        """拒绝立即指向太阳的出发角度。"""
        return not segment_hits_sun(
            src.x, src.y,
            src.x + math.cos(angle) * 5,
            src.y + math.sin(angle) * 5,
            safety=0.5
        )

    def compute_needed(tgt, eta_turns):
        """
        占领 tgt 所需的舰船数，考虑：
          • 驻军增长（航行期间的生产）
          • 在我们之前到达的敌方舰队（可能翻转所有权）
          • 已在飞行中的我方舰船
        """
        early_en = [(t, s) for t, s in inbound_enemy.get(tgt.id, [])
                    if t <= eta_turns]
        if early_en:
            g, _ = garrison_at_arrival(tgt, eta_turns, early_en)
        elif tgt.owner == -1:
            g = tgt.ships                                   # 中立行星不增长
        else:
            g = tgt.ships + tgt.production * eta_turns     # 敌方行星会增长

        g -= committed_to.get(tgt.id, 0)
        return max(0, int(g)) + BUFFER + 1

    def score_target(tgt, eta_turns, needed):
        """
        综合 ROI 评分（越高越好）。
        核心公式：
            (production × productive_turns × 倍率)
            ─────────────────────────────────────
            needed × (1 + eta × 0.018)
        """
        if needed <= 0:
            return 0.0

        prod_turns = max(0, remaining - eta_turns)

        if tgt.id in comet_ids:
            life = comet_life(tgt.id, comets)
            prod_turns = min(prod_turns, max(0, life - eta_turns))
            if prod_turns < COMET_MIN_LIFE:
                return -1.0     # 不值得投入

        base = tgt.production * prod_turns
        if base <= 0 and not is_very_late:
            return -1.0

        mult = 1.0

        # ── 敌方行星 ─────────────────────────────────────────────────
        if tgt.owner not in (player, -1):
            mult *= 1.80        # 我有所得 + 敌有所失（双重剥夺）

            if is_strategic(tgt):
                mult *= 1.25    # 敌方最重要的行星

            # 刚被削弱的：敌方刚从此行星派出舰队
            shipped = enemy_shipped_from.get(tgt.id, 0)
            if shipped > 0:
                weakness = min(0.65, shipped / max(1, tgt.ships + shipped))
                mult *= 1.0 + weakness * 0.9

        # ── 中立行星的竞速条件 ───────────────────────────────────────
        if tgt.owner == -1:
            en_eta = enemy_min_eta(tgt)
            if en_eta < float("inf"):
                ratio = en_eta / max(1.0, float(eta_turns))
                if ratio < 1.35:
                    mult *= 1.45    # 敌人紧随其后——紧急！

        # ── 阶段倍率 ─────────────────────────────────────────────────
        if is_early and tgt.owner == -1 and tgt.id not in comet_ids:
            mult *= 1.50        # 在敌人之前抢占静态/轨道中立行星

        if trailing:
            if tgt.owner not in (player, -1):
                mult *= 1.55    # 落后时必须削减敌方产量
            else:
                mult *= 1.15

        # ── 4 人对战适应 ─────────────────────────────────────────────
        if multiplayer:
            if tgt.owner == -1:
                mult *= 1.20    # 让敌人互相争斗；优先占领免费中立行星
            elif tgt.owner not in (player, -1):
                tgt_str = ships_by.get(tgt.owner, 0)
                min_en  = min((v for k, v in ships_by.items()
                               if k != player), default=1)
                if tgt_str <= min_en * 1.10:
                    mult *= 1.15   # 集中攻击最弱的敌人
                else:
                    mult *= 0.65   # 避免对更强玩家发起两线战争

        # ── 轨道行星奖励（敌人更难瞄准） ─────────────────────────────
        if is_rotating(tgt) and tgt.id not in comet_ids:
            mult *= 1.05

        return (base * mult) / (needed * (1.0 + eta_turns * 0.018))

    def dispatch(src_id, angle, send, tgt_id):
        moves.append([src_id, float(angle), int(send)])
        available[src_id]    -= send
        committed_to[tgt_id]  = committed_to.get(tgt_id, 0) + send

    # ════════════════════════════════════════════════════════════════════
    #  阶段 1 —— 紧急增援
    # ════════════════════════════════════════════════════════════════════
    threatened = []
    for p in my_planets:
        arrivals = fleets_to(p, fleets)
        need     = defense_shortfall(p, arrivals, player)
        already  = committed_to.get(p.id, 0)
        deficit  = need - already
        if deficit > 0:
            roi = p.production * remaining - deficit
            if roi > 0 or p.ships >= 15:
                threatened.append((deficit, p))

    threatened.sort(key=lambda x: -x[0])

    for deficit, at_risk in threatened:
        still = deficit + BUFFER
        helpers = sorted(
            [q for q in my_planets
             if q.id != at_risk.id
             and available[q.id] >= MIN_FLEET
             and pdist(q.x, q.y, at_risk.x, at_risk.y) <= 65],
            key=lambda q: pdist(q.x, q.y, at_risk.x, at_risk.y)
        )
        for helper in helpers:
            if still <= 0:
                break
            send = min(available[helper.id], still)
            if send < MIN_FLEET:
                continue
            res = aim_at(helper, at_risk, send,
                         initial_by_id, ang_vel, comets, comet_ids)
            if res is None:
                continue
            angle, turns, _, _ = res
            if not depart_ok(helper, angle):
                continue
            dispatch(helper.id, angle, send, at_risk.id)
            still -= send

    # ════════════════════════════════════════════════════════════════════
    #  阶段 2 —— 目标优先多源协调攻击
    # ════════════════════════════════════════════════════════════════════

    # 对每个可攻击的行星评分
    scored_targets = []
    for tgt in other_planets:
        if tgt.id in comet_ids and comet_life(tgt.id, comets) < COMET_MIN_LIFE:
            continue

        # 使用最近的有足够兵力的来源来估算 ETA
        best_src = min(
            (p for p in my_planets if available[p.id] >= MIN_FLEET),
            key=lambda p: pdist(p.x, p.y, tgt.x, tgt.y),
            default=None
        )
        if best_src is None:
            continue

        res = aim_at(best_src, tgt, max(MIN_FLEET, available[best_src.id]),
                     initial_by_id, ang_vel, comets, comet_ids)
        if res is None:
            continue
        _, rough_eta, _, _ = res

        if is_late and rough_eta > remaining - 3:
            continue

        need = compute_needed(tgt, rough_eta)
        if need <= 0:
            continue

        sc = score_target(tgt, rough_eta, need)
        if sc <= 0 and not is_very_late:
            continue

        scored_targets.append((sc, tgt))

    scored_targets.sort(key=lambda x: -x[0])

    # 贪心多源分配（按目标优先级顺序）
    for _sc, tgt in scored_targets:
        sources = sorted(
            [p for p in my_planets if available[p.id] >= MIN_FLEET],
            key=lambda p: pdist(p.x, p.y, tgt.x, tgt.y)
        )
        for src in sources:
            avail = available[src.id]
            if avail < MIN_FLEET:
                continue

            res = aim_at(src, tgt, avail,
                         initial_by_id, ang_vel, comets, comet_ids)
            if res is None:
                continue
            angle, turns, _, _ = res

            if not depart_ok(src, angle):
                continue
            if is_late and turns > remaining - 2:
                continue
            if tgt.id in comet_ids and turns >= comet_life(tgt.id, comets):
                continue

            exact_need = compute_needed(tgt, turns)
            if exact_need <= 0:
                break       # 已完全覆盖 —— 停止招募更多来源

            send = max(MIN_FLEET, min(avail, exact_need))
            if send > avail:
                continue

            dispatch(src.id, angle, send, tgt.id)

    # ════════════════════════════════════════════════════════════════════
    #  阶段 3 —— 二次打击：剩余舰船参与部分覆盖
    # ════════════════════════════════════════════════════════════════════
    if not is_very_late:
        for src in my_planets:
            avail = available[src.id]
            if avail < MIN_FLEET * 2:
                continue

            best_sc, best_tgt, best_angle, best_send = -1.0, None, None, 0

            for tgt in other_planets:
                if tgt.id in comet_ids and comet_life(tgt.id, comets) < COMET_MIN_LIFE:
                    continue

                res = aim_at(src, tgt, avail,
                             initial_by_id, ang_vel, comets, comet_ids)
                if res is None:
                    continue
                angle, turns, _, _ = res

                if not depart_ok(src, angle):
                    continue
                if is_late and turns > remaining - 2:
                    continue
                if tgt.id in comet_ids and turns >= comet_life(tgt.id, comets):
                    continue

                need = compute_needed(tgt, turns)
                if need <= 0 or need > avail:
                    continue

                sc = score_target(tgt, turns, need)
                if sc > best_sc:
                    best_sc, best_tgt  = sc, tgt
                    best_angle, best_send = angle, need

            if best_tgt is not None and best_sc > 0:
                send = max(MIN_FLEET, min(available[src.id], best_send))
                if send <= available[src.id]:
                    dispatch(src.id, best_angle, send, best_tgt.id)

    # ════════════════════════════════════════════════════════════════════
    #  阶段 4 —— 前线集结
    # ════════════════════════════════════════════════════════════════════
    if my_planets and enemy_planets and not is_very_late:
        front_d = {mp.id: min(pdist(mp.x, mp.y, e.x, e.y)
                              for e in enemy_planets)
                   for mp in my_planets}
        front   = min(my_planets, key=lambda p: front_d[p.id])

        for rear in my_planets:
            if rear.id == front.id:
                continue
            if front_d[rear.id] < front_d[front.id] * 1.4:
                continue        # 离前线不够远
            avail = available[rear.id]
            send  = int(avail * (0.78 if is_late else 0.62))
            if send < MIN_FLEET:
                continue
            res = aim_at(rear, front, send,
                         initial_by_id, ang_vel, comets, comet_ids)
            if res is None:
                continue
            angle, turns, _, _ = res
            if turns > 50 or not depart_ok(rear, angle):
                continue
            dispatch(rear.id, angle, send, front.id)

    # ════════════════════════════════════════════════════════════════════
    #  阶段 5 —— 终局全力一击（最后 ~35 回合）
    # ════════════════════════════════════════════════════════════════════
    if is_very_late and other_planets:
        for src in my_planets:
            avail = available[src.id]
            if avail < MIN_FLEET:
                continue

            best = None
            for tgt in other_planets:
                res = aim_at(src, tgt, avail,
                             initial_by_id, ang_vel, comets, comet_ids)
                if res is None:
                    continue
                angle, turns, _, _ = res
                if turns > remaining or not depart_ok(src, angle):
                    continue
                sc = (tgt.production * max(0, remaining - turns)
                      - pdist(src.x, src.y, tgt.x, tgt.y) * 0.3)
                if best is None or sc > best[0]:
                    best = (sc, angle, tgt.id)

            if best is None:
                continue
            _, angle, tid = best
            dispatch(src.id, angle, avail, tid)

    return moves